In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('/content/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())

Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [3]:
print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))

Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


In [8]:
import pandas as pd
import plotly.graph_objects as go

# Load dataset
df = pd.read_csv('/content/co2_emissions.csv')

# Filter Asia data
asia_df = df[df['Region'] == 'Asia']

# Choose country to highlight
highlight_country = 'India'   # you can change this

fig = go.Figure()

# Loop through each country
for country in asia_df['Country'].unique():
    country_df = asia_df[asia_df['Country'] == country]

    if country == highlight_country:
        fig.add_trace(go.Scatter(
            x=country_df['Year'],
            y=country_df['CO2_Mt'],
            mode='lines+text',
            name=country,
            line=dict(color='red', width=3),

            # Label at end of line
            text=[None]*(len(country_df)-1) + [country],
            textposition='middle right'
        ))
    else:
        fig.add_trace(go.Scatter(
            x=country_df['Year'],
            y=country_df['CO2_Mt'],
            mode='lines',
            line=dict(color='#DDDDDD', width=1),
            showlegend=False
        ))

# Insight-driven title
fig.update_layout(
    title=f"CO₂ Emissions Trends in Asia — {highlight_country} Shows Rapid Growth Over Time",
    xaxis_title="Year",
    yaxis_title="CO₂ Emissions (Mt)",
    template='plotly_white'
)

fig.show()

In [9]:
import pandas as pd
import plotly.graph_objects as go

# Load dataset
df = pd.read_csv('/content/co2_emissions.csv')

# Aggregate: average CO2 per region per year
region_avg = df.groupby(['Region', 'Year'])['CO2_Mt'].mean().reset_index()

# Filter for 2000 and 2022
slope_df = region_avg[region_avg['Year'].isin([2000, 2022])]

# Pivot for slopegraph
pivot_df = slope_df.pivot(index='Region', columns='Year', values='CO2_Mt').reset_index()
pivot_df.columns = ['Region', 'CO2_2000', 'CO2_2022']

# Sort by 2022 values (better visual order)
pivot_df = pivot_df.sort_values('CO2_2022', ascending=False).reset_index(drop=True)

# Create spacing positions (instead of raw values to avoid overlap)
pivot_df['y_pos'] = range(len(pivot_df))

# Normalize actual values for label display
pivot_df['label_2000'] = pivot_df['CO2_2000'].round(1)
pivot_df['label_2022'] = pivot_df['CO2_2022'].round(1)

fig = go.Figure()

for _, row in pivot_df.iterrows():
    # Softer colors
    color = '#2E86AB' if row['CO2_2022'] > row['CO2_2000'] else '#D64550'

    fig.add_trace(go.Scatter(
        x=[0, 1],
        y=[row['y_pos'], row['y_pos']],
        mode='lines',
        line=dict(color=color, width=3),
        showlegend=False
    ))

    # Left labels (2000)
    fig.add_trace(go.Scatter(
        x=[0],
        y=[row['y_pos']],
        mode='text',
        text=[f"{row['Region']} ({row['label_2000']})"],
        textposition='middle left',
        showlegend=False
    ))

    # Right labels (2022)
    fig.add_trace(go.Scatter(
        x=[1],
        y=[row['y_pos']],
        mode='text',
        text=[f"{row['Region']} ({row['label_2022']})"],
        textposition='middle right',
        showlegend=False
    ))

# Insight title
max_increase = (pivot_df['CO2_2022'] - pivot_df['CO2_2000']).idxmax()
max_decrease = (pivot_df['CO2_2022'] - pivot_df['CO2_2000']).idxmin()

fig.update_layout(
    title=f"Regional CO₂ Change (2000–2022): {pivot_df.loc[max_increase,'Region']} Increased Most, {pivot_df.loc[max_decrease,'Region']} Decreased Most",

    xaxis=dict(
        tickvals=[0, 1],
        ticktext=['2000', '2022'],
        showgrid=False
    ),

    yaxis=dict(
        showticklabels=False,
        showgrid=False
    ),

    template='plotly_white',
    height=500
)

fig.show()